# Raster — JAXAの降水量・標高・土地被覆を検索して表示する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/u-kitazawa/rhinestone/blob/develop/showcase/05_open_raster.ipynb)

```python
results = search(bbox=BBOX, time=TIME_RANGE, providers=[PROVIDER_ID], limit=1)
with resource.open("rasterio", runtime=rasterio) as dataset:
    preview = read_preview(dataset)
```

追加設定・APIキーなしで、日本周辺の公開ラスタを検索し、64×64へ縮小表示します。
**DATASET** を `gsmap`（月降水量）、`aw3d`（標高）、`hrlulc`（土地被覆）から選びます。
**BBOX** の既定は日本周辺（135–145E / 30–40N）。選択したassetが一意でなければ停止します。

各Providerは固定期間・level 0・東経0〜180度の検証済み階層だけを対象とし、
Catalog 1件＋Item 1件の最大2文書を取得します。JAXA全製品・全期間の検索ではありません。
最後にPlanをJSON化し、検索元のSTAC Providerがない別コンテキストで同じ配信を開きます。


## 出典・利用条件

本例はJAXA Earth APIを利用します。
[公式サイトと利用条件](https://data.earth.jaxa.jp/ja/) ·
[研究データ利用条件](https://earth.jaxa.jp/en/data/policy/) ·
[AW3D30条件](https://www.eorc.jaxa.jp/ALOS/aw3d30/terms) ·
[HRLULC条件](https://www.eorc.jaxa.jp/ALOS/lulc/terms.htm) ·
[組み込みProviderの仕様](../docs/api/adapters/stac.md)

| DATASET | データ / asset | 固定期間 | 表示 |
| --- | --- | --- | --- |
| gsmap | GSMaP月降水量 / PRECIP | 2021年2月 | 保存値（mm/hr）、月合計へ換算しない |
| aw3d | AW3D30 v3.2 / DSM | 2021年2月の配信階層 | 標高（m）、観測日時とは区別 |
| hrlulc | HRLULC v25.04 / HRLULC | 2024年 | 土地被覆の分類ID（1〜15）、最近傍で読取 |

登録・APIキーなしの取得は無条件利用を意味しません。STACのライセンスと製品条件を確認し、
出典表示や商用利用前の通知等の適用条件に従ってください。公開成果にはEarth API利用も明記します。
APIとデータは現状有姿で、仕様変更や公開終了の可能性があります。


## 準備

組み込みJAXA Providerを含む固定コミットと、利用者側Runtimeを導入します。
Rhinestone CoreへRasterioやMatplotlibは追加しません。

Colabでは **「ランタイム → すべてのセルを実行」** で先頭から実行してください。
個別実行では、installセル、import・DATASET/BBOXの入力設定セル、プリセット選択セルを順に実行してから検索します。
ランタイムを再起動するとimportと変数定義が失われるため、先頭から実行し直してください。


In [ ]:
import os
import subprocess
import sys

if os.environ.get("RHINESTONE_SHOWCASE_LOCAL") != "1":
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "rhinestone @ git+https://github.com/u-kitazawa/rhinestone.git@bf4b7d98a73f6871b1301c184386abf83cff29a9",
            "rasterio==1.5.2",
            "matplotlib==3.10.7",
        ]
    )
else:
    print("Local smoke: explicit checkout and caller-owned dependencies")

In [ ]:
from datetime import datetime, timezone

from rhinestone import ProviderId, search

DATASET = "gsmap"  # "aw3d" または "hrlulc" へ変更できます
BBOX = (135.0, 30.0, 145.0, 40.0)
PREVIEW_SIZE = 64

In [ ]:
PRESETS = {
    "gsmap": (
        ProviderId.JAXA_GSMAP_PREVIEW,
        "PRECIP",
        2021,
        2,
        28,
        "GSMaP monthly precipitation — February 2021",
        "Blues",
        "Stored value (mm/hr)",
    ),
    "aw3d": (
        ProviderId.JAXA_AW3D_PREVIEW,
        "DSM",
        2021,
        2,
        28,
        "AW3D30 v3.2 elevation preview",
        "terrain",
        "Elevation (m)",
    ),
    "hrlulc": (
        ProviderId.JAXA_HRLULC_PREVIEW,
        "HRLULC",
        2024,
        1,
        31,
        "HRLULC v25.04 — 2024 preview",
        "tab20",
        "Land-cover class ID",
    ),
}
PROVIDER_ID, ASSET_KEY, YEAR, MONTH, LAST_DAY, TITLE, CMAP, VALUE_LABEL = PRESETS[
    DATASET
]
TIME_RANGE = (
    datetime(YEAR, MONTH, 1, tzinfo=timezone.utc),
    datetime(YEAR, MONTH, LAST_DAY, 23, 59, 59, tzinfo=timezone.utc),
)

## 固定された低解像度階層を検索

期間はProviderの固定範囲と一致させます。BBOXはItem取得後の交差判定にも使われます。
0件とProvider障害を区別し、検索成功をRaster本体の取得成功とは扱いません。


In [ ]:
results = None
results = search(
    bbox=BBOX,
    time=TIME_RANGE,
    providers=[PROVIDER_ID],
    limit=1,
)
for diagnostic in results.diagnostics:
    print(
        "Diagnostic:", diagnostic.source_id, diagnostic.reason, diagnostic.failure_type
    )
for candidate in results:
    print(
        candidate.title,
        "|",
        candidate.format,
        "| asset:",
        candidate.reference.parameters.get("asset_key"),
    )
    print("  ", candidate.uri)
if not results:
    if results.diagnostics:
        raise ValueError("Providerの診断を確認してください。データ不在とは限りません。")
    raise ValueError(
        "検索0件です。BBOXと選択Providerの固定期間の交差を確認してください。"
    )

## 選択したデータのassetを明示選択

先頭結果へfallbackしません。Providerとassetの両方が一致し、一意でなければ停止します。


In [ ]:
resource = None
matches = [
    candidate
    for candidate in results
    if candidate.reference.provider_id == PROVIDER_ID
    and candidate.reference.parameters.get("asset_key") == ASSET_KEY
]
if len(matches) != 1:
    raise ValueError(
        f"{ASSET_KEY} assetが検索結果に一意にありません。候補と診断を確認してください。"
    )
resource = matches[0]
print("Selected:", resource.title, resource.format, resource.uri)

## 日本周辺だけを64×64へ縮小読取

RasterioがCOGの必要範囲を読みます。TLS検証を無効化しません。環境で証明書検証に失敗する場合は
管理済みCAをGDALへ設定してください。通信量は配信・GDAL・キャッシュに依存し、64×64の配列サイズと同じではありません。


In [ ]:
import rasterio
from rasterio.enums import Resampling

from rhinestone.errors import ResourceAccessError


def read_preview(dataset):
    if str(dataset.crs) != "EPSG:4326":
        raise ValueError(f"この表示例はEPSG:4326を必要とします: {dataset.crs}")
    return dataset.read(
        1,
        window=dataset.window(*BBOX),
        out_shape=(PREVIEW_SIZE, PREVIEW_SIZE),
        resampling=Resampling.nearest if DATASET == "hrlulc" else Resampling.average,
        masked=True,
    )


preview = None
try:
    with resource.open("rasterio", runtime=rasterio) as dataset:
        preview = read_preview(dataset)
        print("Dataset:", dataset.width, "x", dataset.height, "bands:", dataset.count)
except ResourceAccessError:
    print("Open失敗：配布URL、TLS/CA、Rasterio/GDALの設定を確認してください。")
    raise
if preview.count() == 0:
    raise ValueError("指定範囲に有効な画素がありません。")
print(
    "Preview:",
    preview.shape,
    "valid:",
    preview.count(),
    "min/max:",
    float(preview.min()),
    float(preview.max()),
)

In [ ]:
import matplotlib.pyplot as plt

west, south, east, north = BBOX
figure, axis = plt.subplots(figsize=(7, 6))
image = axis.imshow(
    preview,
    origin="upper",
    extent=(west, east, south, north),
    cmap=CMAP,
    interpolation="nearest",
    **({"vmin": 0.5, "vmax": 15.5} if DATASET == "hrlulc" else {}),
)
axis.set(
    title="JAXA " + TITLE,
    xlabel="Longitude",
    ylabel="Latitude",
)
figure.colorbar(
    image,
    ax=axis,
    label=VALUE_LABEL,
    **({"ticks": list(range(1, 16))} if DATASET == "hrlulc" else {}),
)
figure.text(0.05, 0.01, "Source: JAXA Earth API / " + TITLE, fontsize=8)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

## Plan JSONだけを別コンテキストへ渡す

受信側は、この例で選択した配信URI/形式をdirect Providerで明示許可します。
検索元のSTAC Providerは持たず、CatalogやItemへ再問い合わせしません。
データ本体への通信とRuntimeは必要です。Planにはデータ本体・Credentialは入りません。
ここでは同じkernelの別コンテキストで実証します。外部データが更新されれば値も変わります。


In [ ]:
import json

from rhinestone import AccessPlan, Catalog, Provider, configure

plan_json = json.dumps(configure().plan(resource).to_dict())
received_plan = AccessPlan.from_dict(json.loads(plan_json))
receiving_app = configure(
    catalog=Catalog(
        (
            Provider(
                "permitted-raster",
                "direct",
                {"uri": resource.uri, "format": resource.format},
            ),
        )
    ),
)
plan_preview = None
with receiving_app.open(received_plan, "rasterio", runtime=rasterio) as dataset:
    plan_preview = read_preview(dataset)
if plan_preview.count() == 0:
    raise ValueError("Planの読取範囲に有効画素がありません。")
print("Independent Plan preview:", plan_preview.shape, "valid:", plan_preview.count())

## 境界

- 検索・読取の`BBOX`は`(west, south, east, north)`、Matplotlibの表示範囲は`(west, east, south, north)`へ並べ替えます。
- `BBOX`は日本周辺へ変更できますが、組み込みProviderの経度範囲は東経0〜180度です。
- DATASET変更後は入力設定から後続セルを再実行します。各製品の固定期間・level以外は扱いません。
- 表示値の単位・scale・offsetはSTAC metadataと提供元仕様を確認してください。この例は確認済みの
  保存値を表示し、月合計などへ換算しません。土地被覆の分類IDを平均しません。
- Rasterioの縮小読取とMatplotlibの可視化はRhinestoneではなく利用者側Runtimeの責務です。
- ライブ配信先の可用性は通常CIの必須条件にしません。fixtureで検索・明示選択・縮小読取契約を検証します。
